# DGM-HMamba — demo

<a href="https://colab.research.google.com/github/zarrinmahdavipour/DGM-HMamba/blob/main/notebooks/DGM_HMamba_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Diffusion-guided hierarchical Mamba with dynamic prototype evolution for cross-domain few-shot hyperspectral image classification.**

This notebook:

1. runs **smoke tests on synthetic data** (one forward pass and a few training steps of both stages);
2. downloads **Indian Pines** (source) and **Pavia University** (target);
3. **evaluates a verified model** on its spatially independent test region: 5-way 1-shot and 5-shot episodes and a 9-class classification map, with the same numbers as reported in the README;
4. optionally **trains the verified configuration from scratch** (long).

**Runtime:** about 5–10 minutes for steps 1–3 on a Colab GPU (*Runtime → Change runtime type → T4 GPU*); step 3 takes about 20 minutes on a CPU. Step 4 is off by default (about 8 hours on a 4-core CPU, much less on a GPU).

> The verified results come from a demo-size model (0.50 M parameters) on one transfer task, evaluated on test regions that are geographically separate from the training and validation regions. They are **not** the results reported in the manuscript; see the README.

## 0. Setup

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/zarrinmahdavipour/DGM-HMamba.git"

def run(cmd):
    """Run a shell command, stream its output, and stop the notebook if it fails."""
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"Step failed (exit code {proc.returncode}): {cmd}\n"
                           "The error message is in the output above.")

if os.path.basename(os.getcwd()) == "notebooks":      # opened from a local clone
    os.chdir("..")
if not os.path.isdir("dgm_hmamba"):                     # Colab session
    if not os.path.isdir("DGM-HMamba"):
        run(f"git clone -q {REPO_URL}")
    os.chdir("DGM-HMamba")
run("git pull -q || true")                              # always use the latest code

run("pip -q install -r requirements.txt")
print("Working directory:", os.getcwd())

In [ ]:
import torch
print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Smoke tests on synthetic data (≈1 min)

`demo_forward.py` evaluates the Stage-1 losses (L_cls, L_diff, L_adv) and one Stage-2 episode (L_epi, L_gcl) of a tiny model. At initialisation L_adv ≈ 2·ln 2 ≈ 1.39 and L_epi ≈ ln N, the chance-level values.

In [ ]:
run("python scripts/demo_forward.py")

`demo_train.py` runs a few real optimisation steps of Stage 1 and Stage 2 (random data, so accuracy stays at chance; this only checks that gradients flow through both stages).

In [ ]:
run("python scripts/demo_train.py")

## 2. Data: Indian Pines → Pavia University

Download the two scenes (≈40 MB; from a Hugging Face mirror of the EHU files, with the EHU server as fallback) and check that they load. The source is Indian Pines (AVIRIS, 200 bands, 16 classes) and the target is Pavia University (ROSIS, 103 bands, 9 classes); per-sensor 1×1 band projections map both to a shared 200-channel space.

In [ ]:
run("python scripts/prepare_data.py --data-root data --download indian_pines pavia_u")

## 3. Evaluate a verified model on its test region

The release [`verified-ip2pu-v1`](https://github.com/zarrinmahdavipour/DGM-HMamba/releases/tag/verified-ip2pu-v1) holds the ten models behind the README's verified results (split seeds 5–9 × model seeds 42 and 7). Each file carries its settings and split seed, so `--weights` rebuilds the same geographic partition (64-pixel blocks; training, validation and test regions separated by buffers so that no test patch shares a pixel with a training patch) and evaluates the model without training:

* 5-way 1-shot and 5-shot episodes from the test region;
* a 9-class map: 5 labelled pixels per class from the test region, every other test pixel classified (10 draws).

The model is the manuscript's "w/o Diff" variant (diffusion enhancement bypassed) at demo size; see the README for the full configuration.

In [ ]:
import json, os, urllib.request, zipfile

RUN = "s5_m42"          # s<split seed>_m<model seed>: s5..s9 x m42 / m7
URL = "https://github.com/zarrinmahdavipour/DGM-HMamba/releases/download/verified-ip2pu-v1/verified-ip2pu-v1.zip"
if not os.path.isfile(f"verified/{RUN}/model_final.pt"):
    urllib.request.urlretrieve(URL, "verified-ip2pu-v1.zip")
    zipfile.ZipFile("verified-ip2pu-v1.zip").extractall("verified")
run(f"python scripts/demo_real.py --data-root data --weights verified/{RUN}/model_final.pt --output-dir runs/{RUN}")

In [ ]:
from IPython.display import Image, display

def show(out_dir, recorded=None):
    path = f"{out_dir}/demo_summary.json"
    if not os.path.isfile(path):
        raise RuntimeError("The run did not finish, so there are no results yet. "
                           "Scroll up to the first cell with an error message.")
    s = json.load(open(path))
    if os.path.isfile(f"{out_dir}/demo_maps.png"):
        display(Image(f"{out_dir}/demo_maps.png"))
    print(f"split fingerprint {s['split_info']['index_fingerprint']} | {s['trainable_parameters']/1e6:.2f} M parameters on {s['device']}")
    for name, m in s["episodic_metrics"].items():
        ref = "" if recorded is None else f"   (recorded: OA {recorded['episodic_metrics'][name]['oa_mean']*100:5.2f} %)"
        print(f"{name:>12}: OA {m['oa_mean']*100:5.2f} %   AA {m['aa']*100:5.2f} %   kappa {m['kappa']:.4f}{ref}")
    mm = s.get("map_metrics")
    if mm:
        ref = "" if recorded is None else f"   (recorded: OA {recorded['map_metrics']['oa_mean']*100:5.2f} %)"
        print(f"{'9-class map':>12}: OA {mm['oa_mean']*100:5.2f} ± {mm['oa_sd']*100:.2f} %   "
              f"AA {mm['aa_mean']*100:5.2f} ± {mm['aa_sd']*100:.2f} %   kappa {mm['kappa_mean']:.4f}{ref}")
    return s

summary = show(f"runs/{RUN}", json.load(open(f"verified/{RUN}/demo_summary.json")))

Optional: evaluate all ten models and summarise them as in the README (mean ± SD over the ten runs; about 10 × the time of one evaluation).

In [ ]:
EVALUATE_ALL = False

if EVALUATE_ALL:
    import numpy as np
    rows = []
    for s in (5, 6, 7, 8, 9):
        for m in (42, 7):
            name = f"s{s}_m{m}"
            if not os.path.isfile(f"runs/{name}/demo_summary.json"):
                run(f"python scripts/demo_real.py --data-root data --weights verified/{name}/model_final.pt --output-dir runs/{name}")
            r = json.load(open(f"runs/{name}/demo_summary.json"))
            e, mm = r["episodic_metrics"], r["map_metrics"]
            rows.append([e["5way_1shot"]["oa_mean"], e["5way_1shot"]["aa"], e["5way_5shot"]["oa_mean"],
                         e["5way_5shot"]["aa"], mm["oa_mean"], mm["aa_mean"]])
    a = 100 * np.array(rows)
    for j, k in enumerate(["1-shot OA", "1-shot AA", "5-shot OA", "5-shot AA", "map OA", "map AA"]):
        print(f"{k:>10}: {a[:, j].mean():5.2f} ± {a[:, j].std(ddof=1):.2f}")

## 4. Train the verified configuration yourself (optional, long)

`--preset verified` sets every option of the verified runs: block split with split seed 5, training-only normalisation, 800 Stage-1 iterations, 2,000 Stage-2 episodes with checkpoint selection on the validation region, and the training additions listed in the README. Change `SPLIT_SEED` (5–9) and `SEED` to reproduce another run; any option given after `--preset` overrides it (for example `--stage2-steps 500` for a shorter, less accurate run). Results differ slightly between machines because of floating-point non-determinism.

In [ ]:
TRAIN = False
SPLIT_SEED, SEED = 5, 42

if TRAIN:
    out = f"runs/train_s{SPLIT_SEED}_m{SEED}"
    run(f"python scripts/demo_real.py --data-root data --preset verified --split-seed {SPLIT_SEED} --seed {SEED} --output-dir {out}")
    show(out)
    display(Image(f"{out}/demo_curves.png"))

## 5. Using the library directly

The model can also be used from Python. `encode` runs the diffusion enhancement (DADGE) followed by the hierarchical dual-path Mamba encoder (HDPM); `predict_episode` builds and evolves class prototypes (DSSPE) from a labelled support set and classifies the queries.

In [ ]:
import torch
from dgm_hmamba import DGMHMamba, get_default_config

cfg = get_default_config()            # manuscript hyper-parameters (see dgm_hmamba/config.py)
cfg.hdpm.widths = (32, 32, 32)        # shrink for a quick CPU example
cfg.hdpm.blocks_per_level, cfg.hdpm.state_dim, cfg.hdpm.expand_factor = 1, 4, 1
cfg.dadge.T = cfg.dadge.default_inference_steps = 10

model = DGMHMamba(cfg, source_bands=200, target_bands=103, num_source_classes=16).eval()

n_way, k_shot, n_query = 3, 5, 4
support = torch.rand(n_way * k_shot, 103, 9, 9)          # raw Pavia-U-like patches
query = torch.rand(n_query, 103, 9, 9)
support_y = torch.arange(n_way).repeat_interleave(k_shot)
anchors = torch.randn(n_way, 300)                        # placeholder class-name embeddings

with torch.no_grad():
    sx0 = model.project_bands(support, is_source=False)  # 103 -> 200 channels
    qx0 = model.project_bands(query, is_source=False)
    f = model.encode(qx0)                                # DADGE (full T-step DDPM chain) -> HDPM
    f_fast = model.encode(qx0, num_inference_steps=5, ddim=True)   # accelerated DDIM
    pred = model.predict_episode(sx0, support_y, qx0, n_way, anchors)
print("embedding:", tuple(f.shape), "| predictions:", pred.tolist())

## 6. Full training with `train.py`

`scripts/train.py` runs the complete two-stage protocol with logging, checkpoints, an optional validation pool carved from the target meta-training pool, and a single final evaluation on the meta-testing pool:

```bash
python scripts/train.py --source indian_pines --target pavia_u --data-root data \
    --n-way 5 --k-shot 5 --q-query 15 --stage1-epochs 200 --stage2-steps 100 \
    --seed 42 --output-dir runs/ip2pu_5shot_s42
```

The manuscript-size model needs a large GPU because the selective scan in `mamba_block.py` is a plain PyTorch loop; use `--preset reduced` or `--preset tiny` on smaller hardware. See the README for details and the list of known differences from the manuscript.